In [0]:
-- 1. объём: всего строк и по файлам
SELECT source_file, COUNT(*) AS rows
FROM workspace.shop.bronze_orders
GROUP BY source_file;

In [0]:
-- 2. дубликаты: order_id, который встречается больше 1 раза
SELECT order_id, COUNT(*) AS cnt
FROM workspace.shop.bronze_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
-- 3. NULL / пустые значения по ключевым колонкам
SELECT
  COUNT_IF(order_id    IS NULL OR order_id = '')    AS null_order_id,
  COUNT_IF(customer_id IS NULL OR customer_id = '') AS null_customer_id,
  COUNT_IF(order_ts    IS NULL OR order_ts = '')    AS null_order_ts,
  COUNT_IF(unit_price  IS NULL OR unit_price = '')  AS null_unit_price
FROM workspace.shop.bronze_orders;

In [0]:
-- 4. типы: значения, которые не превращаются в число / дату (в bronze всё string)
SELECT
  COUNT_IF(TRY_CAST(quantity AS INT) IS NULL)          AS bad_quantity,
  COUNT_IF(TRY_CAST(unit_price AS DOUBLE) IS NULL)     AS bad_price,
  COUNT_IF(TRY_TO_TIMESTAMP(order_ts) IS NULL)         AS bad_order_ts
FROM workspace.shop.bronze_orders;

In [0]:
-- 5. бизнес-логика: невозможные значения
SELECT
  COUNT_IF(CAST(quantity AS INT) <= 0)       AS qty_le_0,
  COUNT_IF(CAST(unit_price AS DOUBLE) <= 0)  AS price_le_0
FROM workspace.shop.bronze_orders;

In [0]:
-- 6. справочные значения: нет ли опечаток и неожиданных категорий
SELECT 'currency' AS col, currency AS value, COUNT(*) AS cnt FROM workspace.shop.bronze_orders GROUP BY currency
UNION ALL
SELECT 'status', status, COUNT(*) FROM workspace.shop.bronze_orders GROUP BY status
UNION ALL
SELECT 'country', country, COUNT(*) FROM workspace.shop.bronze_orders GROUP BY country;

In [0]:
-- 7. диапазон дат: нет ли заказов из будущего или из 1900 года
SELECT MIN(order_ts) AS min_ts, MAX(order_ts) AS max_ts
FROM workspace.shop.bronze_orders;

In [0]:
--CREATE OR REFRESH MATERIALIZED VIEW dq_bronze_orders AS
SELECT
  current_timestamp()                                   AS checked_at,        -- когда проверили
  COUNT(*)                                              AS total_rows,        -- всего строк
  COUNT(*) - COUNT(DISTINCT order_id)                   AS duplicate_orders,  -- дубликаты order_id
  COUNT_IF(customer_id IS NULL OR customer_id = '')     AS null_customer,     -- пустой клиент
  COUNT_IF(TRY_TO_TIMESTAMP(order_ts) IS NULL)          AS bad_order_ts,      -- дата не парсится
  COUNT_IF(TRY_CAST(quantity AS INT) IS NULL)           AS bad_quantity_type, -- количество не число
  COUNT_IF(TRY_CAST(quantity AS INT) <= 0)              AS qty_le_0,          -- количество <= 0
  COUNT_IF(TRY_CAST(unit_price AS DOUBLE) <= 0)         AS price_le_0         -- цена <= 0
FROM workspace.shop.bronze_orders;                                            -- таблица из этого же пайплайна